In [1]:
from typing import Any, Optional
from gymnasium.spaces import Sequence, Box, Tuple, Text, Discrete, MultiBinary
from gymnasium import spaces
from gymnasium.wrappers import NormalizeObservation, NormalizeReward
import gymnasium as gym
import numpy as np
from numpy.char import array as chararray
import random
from enum import Enum, auto

import torch as th
import torch.nn as nn

In [2]:
from stable_baselines3.ppo.policies import MlpPolicy, CnnPolicy
from stable_baselines3 import PPO, A2C
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

In [3]:
# DONE: make the action space such that more actions are "valid" manipulations. e.g. not needing to select a particular index rather just selecting from a percentage [0, 1] for the idx.
# DONE: also make actions explicit choices on whether to delete, or insert. 
# DONE: 1. make utility easier to learn, i.e. less discontinuities
# DONE: make everything based off floating points, then round to naturals.
# DONE: normalise the action space 
# DONE: seems like its not learning at all, just doing random actions that don't change state space. 
#   - test after each change to see how it changes things
# DONE: simplified action space is bugged: need fix. it has no way to extend the sequence
# TODO: alter reward space, perhaps make the reward the _difference_ to the last utility?
# TODO: try an _even shorter_ sequence, see if this improves training?

# DONE simplify state space, i.e. shorter sequences like of size 10.
# DONE PARTIALLY simplify action space, i.e. modify one element only at once, or only short sequences. 
# TODO: use autoencoder for the state space
# DONE use an lstm or cnn to process the sequence in the agent

# DONE hand crank the model, see if the action it picks makes sense
# DONE then update the environment, check if the environment is updating properly
# DONE run the model for a few steps if all looks good and see if it approaches the goal.
# DONE write a loop that will run the agent on the environment, see the results rendered.
# DONE perform this on a freshly trained environment.
# DONE to make training more effective, start with a vector of environments with a random initial vector?
# DONE loss is increasing as training happens, learning rate too high?

# TODO: WILD IDEA: reinforcement learning guided by interleaving fuzzing/learning steps. how to implement?
# maybe rollout?

In [4]:
def bytestrings_fixed(l: int) -> gym.Space:
    # Sequence(Box(low=0, high=255, dtype=int), stack=True)
    # return Text(max_length=l, charset=string.hexdigits)
    return Box(low=0, high=255, shape=(l,1), dtype=np.uint8)

def bytestrings_fixed_flat(l: int) -> gym.Space:
    return Box(low=0, high=255, shape=(l,), dtype=np.uint8)
    
def pad_to(a: np.array, l : int) -> np.array:
    '''Returns an array of length `l`, which is `a` padded to exactly
    `l` elements using zeros.
    '''
    e = np.zeros(shape=(l,), dtype=np.uint8)
    pad_len = min(e.size, a.size)
    e[:pad_len] = a[:pad_len]
    return e

def from_padded(a: np.array, pad_elem: int = 0) -> np.array:
    zero_idxs = np.transpose(np.nonzero(a == pad_elem))
    if zero_idxs.size == 0:
        # all doesn't equal `pad_elem` => there are no padding
        return a
    # slice the array until the first zero
    return a[:zero_idxs[0][0]]

def size_of_padded(a: np.array, pad_elem: int = 0) -> int:
    return from_padded(a, pad_elem).size

In [ ]:
PAD_TO = 10

class BitstringEnvFixed(gym.Env):

    def __init__(
            self, 
            init_bytestring: Optional[np.array] = None, 
            max_bytestring_len: int = PAD_TO,
            max_replacement_length: int = 2, 
            step_cost: float = 0.01):
        super().__init__()

        # constant part of the state
        if init_bytestring is None:
            # generate a random array of size 3 to 10, if bytestring not supplied
            arr_size = random.randint(3, 8)
            init_bytestring = np.array([random.randint(1, 255) for _ in range(arr_size)], dtype=np.uint8)
        self._init_bytestring = pad_to(init_bytestring, max_bytestring_len).reshape((max_bytestring_len, 1))

        self.max_bytestring_len = max_bytestring_len
        self.max_replacement_length = max_replacement_length
        self.step_cost = step_cost

        # dynamic part of the state
        self.bytestring = self._init_bytestring
        self.prev_utility = 0.0

    @property
    def observation_space(self):
        return bytestrings_fixed(self.max_bytestring_len)

    @property
    def action_space(self):
        '''Action space:
            - index \in [0, len(bytestring)): O(L)
            - repl_len \in [0, repl_len): O(R)
            - repl_seq \in C ^ repl_len: O(256 ^ repl_len) = O(256 ^ R)
        where L is the max length of the sequence, R is the max length of the replacement
        Total action space O(L * R * 256^R)
        '''
        index_space = Box(0, size_of_padded(self.bytestring), dtype=int)
        replacement_len_space = Box(0, self.max_replacement_length, dtype=int)
        return Tuple(spaces=[
            index_space, replacement_len_space, bytestrings_fixed_flat(self.max_replacement_length)
            ])
    
    @staticmethod
    def unpack_action(action) -> tuple[int, int, np.array]:
        repl_start = action[0][0]
        repl_end = repl_start + action[1][0]
        repl_str = from_padded(action[2])
        return repl_start, repl_end, repl_str

    def reset(self, seed=None, options=None) -> tuple[np.array, dict[str, Any]]:
        super().reset(seed=seed, options=options)
        self.bytestring = self._init_bytestring
        self.prev_utility = 0.0
        return self.bytestring, {}  # empty info dict

    def utility(self) -> float:
        '''Evaluates the current state.
        Abstract method. 
        Should be continuous in the sequence, and measure the distance of the current
        value to the "ideal" one.
        '''
        return 0.0

    def step(self, action) -> tuple[np.array, float, bool, bool, dict[str, Any]]:
        '''Performs the action on the state: a string replacement at the chosen index.
        '''
        # applying action on the state
        repl_start, repl_end, repl_str = self.unpack_action(action)
        self.bytestring = pad_to(
              np.concatenate([
                self.bytestring[:repl_start, 0], repl_str, self.bytestring[repl_end:, 0]])
            , self.max_bytestring_len).reshape((self.max_bytestring_len, 1))

        # Reward based on the utility. Each step has a default negative punishment.
        current_utility = self.utility()
        reward = current_utility - self.prev_utility - self.step_cost
        # reward = 0 - current_utility - self.step_cost
        self.prev_utility = current_utility

        terminated = bool(current_utility == 0)
        truncated = False

        return (
            self.bytestring,
            reward,
            terminated, # bool
            truncated, # bool
            {}, # extra info, dict
        )

    def render(self) -> None:
        # print string representing the environment
        print(from_padded(self.bytestring[:, 0]))

    def close(self):
        pass

In [6]:
class Action(Enum):
    REMOVE: int = 0
    REPLACE: int = 1
    INSERT: int = 2

In [7]:
class BitstringEnvFixedSimpleActions(BitstringEnvFixed):

    @property
    def action_space(self):
        '''Action space:
            - index \in [0, len(bytestring)): O(L)
            - replace or remove: O(2) = O(1)
            - replacement: O(256) = O(1)
            where L is the maximum length of the sequence
            Overall size: O(L * 2 * 256) = O(L * 512) = O(L)
        '''
        index_space = Box(0, size_of_padded(self.bytestring), dtype=int)
        mode = Box(0, 1, dtype=np.uint8)
        replacement = Box(1, 255, dtype=np.uint8)
        return Tuple(spaces=[index_space, mode, replacement])

    @staticmethod
    def unpack_action(action) -> tuple[int, int, np.array]:
        start = action[0][0]
        end = start + 1
        
        mode = Action(action[1][0])
        replacement_chosen = action[2][0]

        replacement = None
        if mode == Action.REMOVE:
            replacement = np.array([], dtype=np.uint8)
        elif mode == Action.REPLACE:
            replacement = np.array([replacement_chosen], dtype=np.uint8)
        elif mode == Action.INSERT:
            replacement = np.array([replacement_chosen], dtype=np.uint8)
            end = start # insert => don't remove original elem

        #print(f"{start}/{replacement}/{end}")

        return start, end, replacement

In [8]:
class BitstringEnvNormalisedActions(BitstringEnvFixed):
    @property
    def action_space(self):
        '''Action space:
            - index \in [-1, 1]: continuous, position on the array
            - replace or remove: [-1, 1]: continuous, <0 => remove, >=0 => replace.
            - replacement \in [0, 1]: continuous, range of values
        '''
        return Box(
            np.array([-1.0, -1.0, -1.0], dtype=np.float32), 
            np.array([ 1.0,  1.0,  1.0], dtype=np.float32), 
            shape=(3,), 
            dtype=np.float32)

    def unpack_action(self, action) -> tuple[int, int, np.array]:
        arr_len = from_padded(self.bytestring).size
        arr_len_half = float(arr_len) / 2.0
        start = int(np.floor(action[0] * arr_len_half + arr_len_half)) # to range [0, arr_len - 1]
        end = start + 1
        
        mode = Action.REPLACE
        if action[1] < -1/3:
            mode = Action.REMOVE
        elif action[1] > 1/3:
            mode = Action.INSERT

        replacement_chosen = np.uint8(np.rint(action[2] * 127 + 128)) # to range [1, 255]
        replacement = None
        if mode == Action.REMOVE:
            replacement = np.array([], dtype=np.uint8)
        elif mode == Action.REPLACE:
            replacement = np.array([replacement_chosen], dtype=np.uint8)
        elif mode == Action.INSERT:
            replacement = np.array([replacement_chosen], dtype=np.uint8)
            end = start # insert => don't remove original elem
            
        return start, end, replacement

In [9]:
TARGET = [31, 41, 59, 26, 54, 27, 171]

In [10]:
class HammingEnv(BitstringEnvFixedSimpleActions):
    @staticmethod
    def hamming_d(str_a: np.array, str_b: np.array) -> float:
        '''Distance function: dist(str_a, str_b) = | len(a) - len(b) | + | mismatches |
        '''
        flat_a, flat_b = str_a.flatten(), str_b.flatten()
        length_d = abs(flat_a.size - flat_b.size)
        compare_upto = flat_a.size
        if length_d != 0:
            compare_upto = min(flat_a.size, flat_b.size)

        mismatches = 0
        for i in range(compare_upto):
            mismatches += 1 if flat_a[i] != flat_b[i] else 0

        return 10 * length_d + mismatches

    def utility(self) -> float:
        target = np.array(TARGET, dtype=np.uint8)
        source = from_padded(self.bytestring.flatten())
        return HammingEnv.hamming_d(source, target)

In [21]:
class EuclideanEnv(BitstringEnvNormalisedActions):
    @staticmethod
    def euclidean_d(str_a: np.array, str_b: np.array) -> float:
        shorter, longer = (str_a, str_b) if str_a.size < str_b.size else (str_b, str_a)
        shorter_padded = pad_to(shorter, longer.size)
        return np.sqrt(np.sum((longer - shorter_padded) ** 2))

    def utility(self) -> float:
        target = np.array(TARGET, dtype=np.uint8)
        source = from_padded(self.bytestring.flatten())

        length_loss = abs(target.size - source.size)

        return length_loss * 30 + EuclideanEnv.euclidean_d(source, target)

In [12]:
# Test the simplified actions environment
test_env = HammingEnv(np.array([1, 2, 3, 4, 5], dtype=np.uint8))
test_env.render() # expected [1, 2, 3, 4, 5]

# replace element 3 with 69
print("replacing element #3 with 69.")
ob, rew, _, _, _ = test_env.step(([2], [1], np.array([69], dtype=np.uint8)))
test_env.render() # expected [1, 2, 69, 4, 5]

# remove element 1 and 2
print("removing element 1 and 2")
test_env.step(([0], [0], np.array([5], dtype=np.uint8)))
test_env.step(([0], [0], np.array([7], dtype=np.uint8)))
test_env.render() # expected [69, 4, 5]

# adding to the 2nd element of the list
print("adding between position 1 and 2")
test_env.step([[1], [2], np.array([77], dtype=np.uint8)])
test_env.render() # expected [69, 77, 4, 5]

[1 2 3 4 5]
replacing element #3 with 69.
[ 1  2 69  4  5]
removing element 1 and 2
[69  4  5]
adding between position 1 and 2
[69 77  4  5]


In [13]:
# Test for continuous environment (normalised)
test_env2 = EuclideanEnv(np.array([1, 2, 3, 4, 5], dtype=np.uint8))
test_env2.render() # expected [1, 2, 3, 4, 5]

# replace element 4 with 42
print("replacing element #4 with 42")
test_env2.step(np.array([0.40, 0.10, (41.5 / 255.0) * 2.0 - 1.0]))
test_env2.render() # expected [1, 2, 3, 42, 5]

# remove elements 2 and 4
print("removing element #2")
test_env2.step(np.array([-0.60, -0.50, 0.00]))
test_env2.render() # expected [1, 3, 42, 5]

print("removing element #3 (was #4)")
test_env2.step(np.array([0.25, -0.40, 0.69]))
test_env2.render() # expected [1, 3, 5]

print("adding 77 between index #2 and #3")
test_env2.step(np.array([0.50, 0.75, (76.5 / 255.0) * 2.0 - 1.0]))
test_env2.render() # expected [1, 3, 77, 5]

[1 2 3 4 5]
replacing element #4 with 42
[ 1  2  3 42  5]
removing element #2
[ 1  3 42  5]
removing element #3 (was #4)
[1 3 5]
adding 77 between index #2 and #3
[ 1  3 77  5]


In [14]:
class FlattenAction(gym.ActionWrapper):
    """Action wrapper that flattens the action."""
    def __init__(self, env):
        super(FlattenAction, self).__init__(env)
        self.action_space = gym.spaces.utils.flatten_space(self.env.action_space)
        
    def action(self, action):
        return gym.spaces.utils.unflatten(self.env.action_space, action)

    def reverse_action(self, action):
        return gym.spaces.utils.flatten(self.env.action_space, action)

In [15]:
class ReshapeChannelFirst(gym.ObservationWrapper):
    def __init__(self, env):
        '''Assumes original observation is a Box of shape (n, 1)'''
        super(ReshapeChannelFirst, self).__init__(env)
        obs = self.observation_space
        (n, _), dtype, low, high = obs.shape, obs.dtype, obs.low.T, obs.high.T
        self.observation_space = Box(low=low, high=high, shape=(1, n), dtype=dtype)

    def observation(self, observation: np.array):
        '''Assumes observation is a 1d array of shape (length, 1)'''
        size = observation.size
        return observation.reshape((1, size))
    
    def reverse_observation(self, t_observation: np.array):
        '''Again assumes original observation has shape (size, 1)'''
        size = t_observation.size
        return t_observation.reshape((size, 1))

In [16]:
env = gym.wrappers.TimeLimit(
    FlattenAction(
    ReshapeChannelFirst(
    NormalizeReward(
    NormalizeObservation(
    EuclideanEnv(init_bytestring=np.array([8] * 6, dtype=np.uint8)))))),
    100
    )
check_env(env)

/home/jacob/.local/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:272: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(


In [17]:
wrapped_env_ctor = lambda **kwargs: gym.wrappers.TimeLimit(
                                        FlattenAction(
                                        ReshapeChannelFirst(
                                        NormalizeObservation(
                                        EuclideanEnv(**kwargs)))), 30)
vec_env = make_vec_env(wrapped_env_ctor, n_envs=3)

In [18]:
class CustomCNN(BaseFeaturesExtractor):
    """
    :param observation_space: (gym.Space)
    :param features_dim: (int) Number of features extracted.
        This corresponds to the number of unit for the last layer.
    """

    def __init__(self, observation_space: spaces.Box, features_dim: int = 256):
        super().__init__(observation_space, features_dim)
        # We assume CxHxW images (channels first)
        # Re-ordering will be done by pre-preprocessing or wrapper
        n_input_channels = observation_space.shape[0]
        self.cnn = nn.Sequential(
            # original architecture on 
            # https://stable-baselines3.readthedocs.io/en/master/guide/custom_policy.html#on-policy-algorithms
            nn.Conv1d(n_input_channels, 16, kernel_size=3, stride=2, padding=0),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=3, stride=1, padding=0),
            nn.ReLU(),
            nn.Flatten(),
        )

        # Compute shape by doing one forward pass
        with th.no_grad():
            n_flatten = self.cnn(
                th.as_tensor(observation_space.sample()[None]).float()
            ).shape[1]

        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())

    def forward(self, observations: th.Tensor) -> th.Tensor:
        return self.linear(self.cnn(observations))

policy_kwargs = dict(
    features_extractor_class=CustomCNN,
    features_extractor_kwargs=dict(features_dim=128),
)

In [ ]:
model = PPO(
    MlpPolicy,
    # CnnPolicy, 
    vec_env, 
    verbose=1, 
    learning_rate=3e-4,
    # policy_kwargs=policy_kwargs,
    policy_kwargs=dict(net_arch=[64, 64]), 
    tensorboard_log="./logs/ppo_mlp_int_repr")
model.learn(total_timesteps=5_000_000, tb_log_name="incremental_euclideanloss_normalised_simpleact_lengthloss3_seq10_envs3_mlp6464_rate1e-3_timelimit30")

Using cpu device
Logging to ./logs/ppo_mlp_int_repr/euclideanloss_normalised_simpleact_shorterseq_envs3_mlp6464_rate1e-3_timelimit30_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 30       |
|    ep_rew_mean     | -884     |
| time/              |          |
|    fps             | 2223     |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 6144     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 30           |
|    ep_rew_mean          | -881         |
| time/                   |              |
|    fps                  | 1586         |
|    iterations           | 2            |
|    time_elapsed         | 7            |
|    total_timesteps      | 12288        |
| train/                  |              |
|    approx_kl            | 0.0068417527 |
|    clip_fraction        | 0.0692       |
|    clip

In [57]:
def apply_action(cls, obs: np.array, action: np.array) -> np.array:
    action_int = np.floor(action).astype(int) # floor is used instead of round
    unflattened_action = FlattenAction(cls()).action(action_int)
    mut_idx, mut_substr_length, mut_replacement = cls.unpack_action(unflattened_action)
    mut_obs = pad_to(
          np.concatenate([obs[0, :mut_idx], mut_replacement, obs[0, mut_idx+mut_substr_length:]])
        , PAD_TO)
    return mut_obs

In [20]:
env = wrapped_env_ctor()
ob, _ = env.reset()
print("initial:")
env.render()

for i in range(30):
    act, _ = model.predict(ob)
    # ob_alt = apply_action(EuclideanEnv, ob, act)
    ob, rwd, trm, tnc, _ = env.step(act)
    print(f"step {i+1}: applied {act=}, reward {rwd=}")

    # all_equal = np.all(ob_alt == ob[0, :])
    # print(f"{all_equal=}")
    # if not all_equal:
    #     print(f"alt: {from_padded(ob_alt)}")
    #     print(f"obs: {from_padded(ob)}")
    
    env.render()
    
# TARGET = [31, 41, 59, 26, 54, 27, 171] 

initial:
[175  93 152]
step 1: applied act=array([-0.77538043,  0.19199017, -0.11357364], dtype=float32), reward rwd=np.float64(-33.41658617698013)
[114  93 152]
step 2: applied act=array([-1.        ,  0.24868831,  1.        ], dtype=float32), reward rwd=np.float64(-29.725315916207254)
[255  93 152]
step 3: applied act=array([ 0.40968138, -1.        , -0.6938779 ], dtype=float32), reward rwd=np.float64(-28.90636655359978)
[255  93]
step 4: applied act=array([ 1.        , -0.73137426, -0.5470618 ], dtype=float32), reward rwd=np.float64(-28.90636655359978)
[255  93]
step 5: applied act=array([ 1.        , -1.        , -0.14190513], dtype=float32), reward rwd=np.float64(-28.90636655359978)
[255  93]
step 6: applied act=array([ 0.72830963, -0.79207426, -0.5321038 ], dtype=float32), reward rwd=np.float64(-28.923664589601923)
[255]
step 7: applied act=array([ 1.        , -0.31520742,  1.        ], dtype=float32), reward rwd=np.float64(-30.325012782448237)
[255 255]
step 8: applied act=array

In [22]:
rwd_mean, rwd_std = evaluate_policy(model, vec_env)
print(f"{rwd_mean=}, {rwd_std=}")

rwd_mean=np.float64(-2.1065826999999997), rwd_std=np.float64(0.31435795818940226)
